[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/12_linear_attention.ipynb)

# 🔴 Hard: Linear Self-Attention

Implement **Linear Attention** — O(S·D²) instead of O(S²·D), enabling efficient long-sequence processing.

Replace softmax with a **kernel feature map** $\phi$:

$$\text{LinearAttn}(Q,K,V) = \frac{\phi(Q) \left(\phi(K)^T V\right)}{\phi(Q) \cdot \sum \phi(K)}$$

### Feature map
Use $\phi(x) = \text{elu}(x) + 1$ (ensures non-negative features).

### Signature
```python
def linear_attention(Q, K, V):
    # Q: (B, S, D_k), K: (B, S, D_k), V: (B, S, D_v)
    # Returns: (B, S, D_v)
```

### Key insight
Instead of computing the $S \times S$ attention matrix, compute $\phi(K)^T V$ first (a $D_k \times D_v$ matrix), then multiply by $\phi(Q)$.

### Rules
- Must use a feature map (NOT softmax)
- Must be O(S·D²) — should run fast on long sequences
- You **may** use `F.elu`

### My notes:
#### Why Linear Attention is O(SD²)?

Suppose:

$$
K^T: (D, S)
$$

$$
V: (S, D)
$$

Then:

$$
K^T V: (D, S)(S, D) \rightarrow (D, D)
$$

The output has $D^2$ elements.

Each output element is an inner product of length $S$:

$$
(K^T V)*{ij} = \sum*{s=1}^{S} K_{s,i}V_{s,j}
$$

So:

* Number of output elements = $D^2$
* Cost per output element = $S$

Therefore:

$$
\text{Complexity} = S \times D^2 = O(SD^2)
$$



In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import torch.nn.functional as F

In [36]:
# ✏️ YOUR IMPLEMENTATION HERE

import math

def linear_attention(Q, K, V):
    # pass  # Replace this
    Q = F.elu(Q) + 1
    K = F.elu(K) + 1
    return (Q @ (K.transpose(-2, -1) @ V)) / (Q @ (K.transpose(-2, -1).sum(dim=-1, keepdim=True)))
    # return torch.softmax(Q @ K.transpose(-2, -1) / math.sqrt(Q.size(-1)), dim=-1) @ V

    

In [37]:
# 🧪 Debug
Q = torch.randn(1, 8, 16)
K = torch.randn(1, 8, 16)
V = torch.randn(1, 8, 32)
out = linear_attention(Q, K, V)
print("Output shape:", out.shape)   # (1, 8, 32)
print("Has NaN?", torch.isnan(out).any().item())

Output shape: torch.Size([1, 8, 32])
Has NaN? False


In [38]:
from torch_judge import check
check('linear_attention')


🧪 Testing: Linear Self-Attention (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shape (0.6ms)
  ✅ [2/4] No NaN or Inf (6.0ms)
  ✅ [3/4] Gradient flow (0.5ms)
  ✅ [4/4] Runs fast on long sequences (linear complexity) (4.4ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (11.4ms total)
  Progress saved. Run status() to see your dashboard.

